# Deployment giả lập — dự báo household purchase propensity

Mục tiêu là minh họa quy trình từ dữ liệu giao dịch đến danh sách xác suất tại cutoff. Pipeline POC đã chốt là LightGBM với 18 feature và các feature chu kỳ; phiên bản này được fit trên cutoff đến ngày 439 và giữ xác suất thô theo lựa chọn calibration trong train. Không refit hay lựa chọn mô hình theo test.

Tại cutoff 711, dự báo xác suất household có **ít nhất một basket trong ngày 712–718** dựa trên dữ liệu đến ngày 711. Nguồn kết thúc tại ngày 711, vì vậy không có nhãn trưởng thành để kiểm chứng dự báo mới nhất. Đây là scoring giả lập, không phải triển khai production.

## 1. Nạp pipeline và dữ liệu

Chỉ nạp artifact nội bộ do project tạo. Dữ liệu gốc được giữ nguyên; `QUANTITY`, demographic và marketing không tham gia. Các feature phải khớp đúng tên và thứ tự trong pipeline đã lưu.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.special import logit
import joblib
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
if (ROOT/'.deps').is_dir():sys.path.insert(0,str(ROOT/'.deps'))
from propensity_v2.prepared_data import load_prepared_split
OUT=ROOT/'artifacts'/'deployment_poc'
package=joblib.load(ROOT/'artifacts'/'calibration_final_poc'/'fixed_pipelines.joblib')
bundle=package['bundles']['18 + chu kỳ'];features=bundle['features']
assert len(features)==23 and package['fit_cutoffs']==[180,439]
tx=pd.read_csv(ROOT/'sample_data'/'dunnhumby'/'transaction_data.csv',
               usecols=['household_key','BASKET_ID','DAY','SALES_VALUE'])
assert not tx.isna().any().any()
print('Dữ liệu đến ngày:',tx.DAY.max(),'| Số feature:',len(features),'| Calibration:', 'sigmoid' if bundle['calibrator'] is not None else 'raw')

Dữ liệu đến ngày: 711 | Số feature: 23 | Calibration: raw


## 2. Tính feature tại một cutoff

Mỗi household đủ điều kiện có basket được ghi nhận đến cutoff. Tổng hợp basket theo household/ngày; tính lịch sử trong `(s-w,s]` cho `w = 7, 30, 90, 180`. Các feature chu kỳ dùng khoảng cách giữa ngày có basket trong `(s-180,s]`. Thiếu lịch sử không được giả thành một khoảng cách bằng 0.

In [2]:
def make_snapshot(transactions,cutoff):
    source=transactions.loc[transactions.DAY<=cutoff].copy()
    if source.empty:raise ValueError('Không có giao dịch trước cutoff')
    scope=source.groupby('BASKET_ID').agg(households=('household_key','nunique'),days=('DAY','nunique'))
    if not (scope.households.eq(1).all() and scope.days.eq(1).all()):
        raise ValueError('Basket ID không duy nhất theo household và ngày')
    basket=(source.groupby(['household_key','BASKET_ID','DAY'],as_index=False,sort=False)
            .agg(sales=('SALES_VALUE','sum'),line_items=('SALES_VALUE','size')))
    daily=(basket.groupby(['household_key','DAY'],as_index=False)
           .agg(baskets=('BASKET_ID','size'),sales=('sales','sum'),line_items=('line_items','sum')))
    first=daily.groupby('household_key').DAY.min()
    last=daily.groupby('household_key').DAY.max()
    snapshot=pd.DataFrame({'household_key':first.index,'snapshot_day':cutoff,
                           'recency_days':cutoff-last.to_numpy(),
                           'observed_tenure_days':cutoff-first.to_numpy()}).set_index('household_key')
    for w in (7,30,90,180):
        window=daily.loc[(daily.DAY>cutoff-w)&(daily.DAY<=cutoff)]
        agg=window.groupby('household_key').agg(baskets=('baskets','sum'),sales=('sales','sum'),
                                                line_items=('line_items','sum'),active_days=('DAY','size'))
        for col in ['baskets','sales','line_items','active_days']:
            snapshot[f'{col}_{w}d']=agg[col].reindex(snapshot.index).fillna(0)
    # Match Data Preparation's cumulative float64 arithmetic exactly. Small rounding
    # differences near a tree split can otherwise change predicted probabilities.
    sales_matrix=(daily.pivot(index='household_key',columns='DAY',values='sales')
                  .reindex(index=snapshot.index,columns=range(cutoff+1)).fillna(0)
                  .to_numpy(dtype=np.float64))
    cumulative_sales=np.cumsum(sales_matrix,axis=1)
    for w in (7,30,90,180):
        snapshot[f'sales_{w}d']=cumulative_sales[:,cutoff]-cumulative_sales[:,max(cutoff-w,0)]
    days=daily.loc[(daily.DAY>cutoff-180)&(daily.DAY<=cutoff),['household_key','DAY']].sort_values(['household_key','DAY']).copy()
    days['gap']=days.groupby('household_key').DAY.diff()
    gap=days.groupby('household_key').gap.agg(['count','median','std'])
    snapshot['cadence_median_gap_180d']=gap['median'];snapshot['cadence_gap_std_180d']=gap['std']
    snapshot['cadence_has_gap']=gap['count'].ge(1).reindex(snapshot.index,fill_value=False).astype(int)
    snapshot['cadence_has_two_gaps']=gap['count'].ge(2).reindex(snapshot.index,fill_value=False).astype(int)
    snapshot['cadence_recency_ratio']=snapshot.recency_days/snapshot.cadence_median_gap_180d
    snapshot=snapshot.reset_index()
    assert not snapshot.duplicated(['household_key','snapshot_day']).any()
    assert set(features).issubset(snapshot.columns)
    assert not snapshot[features].isin([np.inf,-np.inf]).any().any()
    return snapshot
def score_snapshot(snapshot):
    p=bundle['model'].predict_proba(snapshot[features])[:,1]
    if bundle['calibrator'] is not None:
        p=bundle['calibrator'].predict_proba(logit(np.clip(p,package['epsilon'],1-package['epsilon'])).reshape(-1,1))[:,1]
    assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
    return p

## 3. Kiểm tra đối chiếu tại cutoff có dữ liệu tham chiếu

Tính lại feature tại cutoff validation 551 từ giao dịch nguồn, đối chiếu 18 feature với bảng chuẩn bị và dự báo với artifact đã lưu. Chỉ kiểm tra phép tính; không dùng nhãn validation để chọn model hay điều chỉnh pipeline.

In [3]:
validation,_=load_prepared_split(ROOT/'artifacts'/'data_preparation','validation')
reference=validation.loc[validation.snapshot_day.eq(551)]
check=make_snapshot(tx,551)
joined=reference.merge(check,on=['household_key','snapshot_day'],validate='one_to_one',suffixes=('_prepared','_scoring'))
assert len(joined)==len(reference)==len(check)
base_features=[c for c in features if not c.startswith('cadence_')]
for col in base_features:
    assert np.allclose(joined[col+'_prepared'],joined[col+'_scoring'],rtol=1e-10,atol=1e-7),col
check['probability']=score_snapshot(check)
saved=pd.read_parquet(ROOT/'artifacts'/'calibration_final_poc'/'predictions.parquet',
                      filters=[('split','=','validation'),('feature_set','=','18 + chu kỳ'),('snapshot_day','=',551)])
compare=check.merge(saved[['household_key','snapshot_day','probability']],on=['household_key','snapshot_day'],validate='one_to_one',suffixes=('_new','_saved'))
assert len(compare)==len(check) and np.allclose(compare.probability_new,compare.probability_saved,rtol=0,atol=1e-12)
print('PASS:',len(check),'household, 18 feature cơ bản và xác suất khớp tham chiếu tại cutoff 551.')

PASS: 2498 household, 18 feature cơ bản và xác suất khớp tham chiếu tại cutoff 551.


## 4. Scoring tại ngày mới nhất

Tính xác suất có basket trong 7 ngày kế tiếp cho mỗi household đã xuất hiện. Thứ hạng được sắp theo xác suất giảm dần, phá đồng điểm bằng `household_key` tăng dần để tái lập; thứ hạng không phải quyết định liên hệ hay gửi ưu đãi.

In [4]:
cutoff=int(tx.DAY.max())
latest=make_snapshot(tx,cutoff)
latest['purchase_probability_next_7d']=score_snapshot(latest)
scores=latest[['household_key','snapshot_day','purchase_probability_next_7d']].sort_values(
    ['purchase_probability_next_7d','household_key'],ascending=[False,True]).reset_index(drop=True)
scores['rank']=np.arange(1,len(scores)+1)
assert cutoff==711 and scores.household_key.nunique()==len(scores)
print('Cutoff:',cutoff,'| Household:',len(scores),'| Khoảng dự báo:',cutoff+1,'–',cutoff+7)
display(scores.purchase_probability_next_7d.describe().rename('xác suất dự báo').to_frame())

Cutoff: 711 | Household: 2500 | Khoảng dự báo: 712 – 718


,xác suất dự báo
count,2500.000000
mean,0.525145
std,0.296582
min,0.044894
25%,0.257009
50%,0.526344
75%,0.799540
max,0.998163


## 5. Xuất dữ liệu và kế hoạch theo dõi

File scoring chỉ là đầu ra POC. Khi có dữ liệu mới, cần kiểm tra độ mới của nguồn, khóa/basket và feature thiếu; so sánh phân bố recency, số ngày hoạt động và xác suất với các giai đoạn trước. Khi nhãn 7 ngày đã trưởng thành, theo dõi Brier, log loss, ROC-AUC, calibration và lift theo cutoff; điều tra sự suy giảm trước khi quyết định cập nhật mô hình. Các ngưỡng cảnh báo cụ thể chưa được hiệu chuẩn cho hệ thống thực tế.

Vì chỉ số `DAY` chưa ánh xạ ngày thực và nguồn kết thúc ở 711, không diễn giải kết quả như một dự báo lịch thực hiện tại. Mô hình dự báo giao dịch được ghi nhận, không đo tác động của liên hệ hay coupon.

In [5]:
OUT.mkdir(parents=True,exist_ok=True)
scores.to_csv(OUT/'household_scores_day_711.csv',index=False,encoding='utf-8-sig')
print('Đã xuất',len(scores),'dòng:',OUT/'household_scores_day_711.csv')

Đã xuất 2500 dòng: artifacts/deployment_poc/household_scores_day_711.csv
